# 09 Final Artifact Audit

Generated from `notebooks/ledgar_clause_classification_pipeline.ipynb`.

Source cell indices: `51, 52`.

- Run this after experiments or after rebuilding report artifacts.
- It checks evidence files and report-facing figures/tables.


## 1. Colab Setup, Imports, and Configuration

This stage prepares the runtime so the same notebook can run locally or in Google Colab. In Colab, upload, unzip, sync, or clone the whole project folder, not just this notebook.



Importing Libraries and Modules

In [ ]:
from pathlib import Path

import importlib.util
import os
import subprocess
import sys
import numpy as np
import pandas as pd

import json
import math
import re

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

File Setup

In [ ]:
PROJECT_ROOT_OVERRIDE = os.environ.get("LEDGAR_PROJECT_ROOT", "").strip()
AUTO_MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS_IN_COLAB = True


def running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or importlib.util.find_spec("google.colab") is not None


IN_COLAB = running_in_colab()

In [ ]:

if IN_COLAB:
    print("Google Colab runtime detected.")

if IN_COLAB and AUTO_MOUNT_GOOGLE_DRIVE:
    try:
        if Path("/content/drive/MyDrive").exists():
            print("Google Drive is already available.")
        else:
            from google.colab import drive

            drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped/failed: {type(exc).__name__}: {exc}")


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "modules").is_dir()


def project_root_candidates_near(path: Path) -> list[Path]:
    path = path.expanduser()
    candidates = [path, *path.parents]
    if path.exists() and path.is_dir():
        for pattern in (
            "pyproject.toml",
            "*/pyproject.toml",
            "*/*/pyproject.toml",
            "*/*/*/pyproject.toml",
        ):
            candidates.extend(pyproject.parent for pyproject in path.glob(pattern))
    deduped = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved not in seen:
            deduped.append(resolved)
            seen.add(resolved)
    return deduped


def parent_search(start: Path) -> Path | None:
    for candidate in project_root_candidates_near(start):
        if looks_like_project_root(candidate):
            return candidate
    return None


def common_colab_candidates() -> list[Path]:
    candidates = [
        Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing"),
    ]
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            for pattern in (
                "Natural-Language-Processing",
                "*/Natural-Language-Processing",
                "*/*/Natural-Language-Processing",
                "*/*/*/Natural-Language-Processing",
            ):
                candidates.extend(base.glob(pattern))
    return candidates


def find_notebook_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        for candidate in project_root_candidates_near(override):
            if looks_like_project_root(candidate):
                if candidate != override:
                    print(f"PROJECT_ROOT_OVERRIDE pointed to a parent folder; using nested project root: {candidate}")
                return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain pyproject.toml and modules/, and no nested project root was found under it: {override}\n"
            "Check the Drive folder path, or run this diagnostic: list(Path('/content/drive/MyDrive').glob('**/pyproject.toml'))"
        )

    root = parent_search(Path.cwd())
    if root is not None:
        return root

    if IN_COLAB:
        for candidate in common_colab_candidates():
            if candidate.exists() and looks_like_project_root(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml and modules/.\n"
        "In Colab, upload or clone the whole repository, then set PROJECT_ROOT_OVERRIDE "
        "near the top of this cell to that folder. Current working directory: "
        f"{Path.cwd()}"
    )

In [ ]:
# Find the project root and add it to sys.path so that imports work, even if the notebook is opened in a subfolder or outside the project.
PROJECT_ROOT = find_notebook_project_root()
os.environ["LEDGAR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# List of (import_name, pip_name) for packages commonly used in notebooks. pip_name can be None if it's the same as import_name.
REQUIRED_NOTEBOOK_PACKAGES = [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]

# In Colab, install all requirements from requirements-colab.txt if any are missing, to avoid multiple pip installs.
def ensure_notebook_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

# Check for missing imports before installing requirements in Colab, to avoid unnecessary pip installs and speed up notebook startup.
missing_imports = [name for name, _ in REQUIRED_NOTEBOOK_PACKAGES if importlib.util.find_spec(name) is None]
requirements_path = PROJECT_ROOT / "requirements-colab.txt"


if IN_COLAB and INSTALL_REQUIREMENTS_IN_COLAB and requirements_path.exists() and missing_imports:
    print(f"Installing Colab requirements from {requirements_path}.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
else:
    for import_name, pip_name in REQUIRED_NOTEBOOK_PACKAGES:
        ensure_notebook_package(import_name, pip_name)

Custom Modules and Libraries

In [ ]:
from modules.data_setup import (
    adapt_cuad_to_clause_classification,
    build_project_paths,
    download_cuad_if_missing,
    load_cuad_raw_files,
    load_or_download_ledgar,
    print_dataset_availability,
    seed_everything,
)
from modules.preprocessing import create_ledgar_eda, preprocess_ledgar
from modules.baselines import run_baseline_experiments
from modules.classical_models import run_classical_experiments
from modules.sequence_model import SequenceModelConfig, train_sequence_classifier
from modules.transformer_model import train_transformer_classifier
from modules.transformer_hpt import TransformerHPTConfig, run_two_stage_transformer_hpt
from modules.qwen_prompting import run_qwen_baseline
from modules.agentic_review import run_agentic_review
from modules.evaluation import save_final_comparison
from modules.error_analysis import run_error_analysis
from modules.wandb_reporting import finish_wandb_run, log_wandb_outputs, start_wandb_run



| Setting | Value | Purpose |
|---|---:|---|
| `SEED` | `42` | Makes sampling, baseline randomness, and train/test helper behavior reproducible. |
| `DATASET_NAME` | `LEDGAR` | Keeps the main experiment scoped to LEDGAR clause classification. |
| `TOP_K_LABELS` | `20` | Restricts the task to the 20 most frequent training labels for a manageable coursework experiment. |
| `RUN_CLASSICAL_MODELS` | `True` | Enables TF-IDF model experiments. |
| `RUN_TRANSFORMER` | `True` | Attempts transformer fine-tuning only when the runtime can support it. |
| `RUN_QWEN_BASELINE` | `True` | Attempts Qwen prompting only when GPU/model loading is available. |
| `RUN_AGENTIC_EXTENSION` | `True` | Enables a small review workflow demonstration, not an autonomous agent. |
| `RUN_WANDB` | `True` | Sends metrics and safe artifacts to W&B when credentials are available. |

Model and feature hyperparameters declared here:

| Component | Hyperparameters |
|---|---|
| TF-IDF search | `max_features` in `[10000, 30000]`; `ngram_range` in `[(1, 1), (1, 2)]`; `lowercase=True`; `stop_words=None` |
| Transformer | `distilbert-base-uncased`; `max_length=256` |
| Optional legal transformer | `nlpaueb/legal-bert-base-uncased` can be substituted manually if GPU resources allow |
| Qwen prompting | `Qwen/Qwen2.5-3B-Instruct`; test sample size `200`; one few-shot example per class when available |
| W&B logging | Uses `WANDB_API_KEY` from Colab Secrets or the environment; text-containing prediction/error tables are not uploaded unless `WANDB_LOG_TEXT_TABLES=True` |

Explainability note: keeping all configuration values in one cell makes it clear which choices affect runtime cost, model capacity, and evaluation scope.

In [ ]:
SEED = 42

DATASET_NAME = "LEDGAR"

TOP_K_LABELS = 20

MAX_FEATURES_LIST = [10000, 30000]

NGRAM_RANGES = [(1, 1), (1, 2)]

RUN_CLASSICAL_MODELS = True

RUN_TRANSFORMER = True

RUN_TRANSFORMER_HPT = False

HPT_RANDOM_TRIALS = 8

HPT_BAYES_TRIALS = 8

RUN_QWEN_BASELINE = True

RUN_AGENTIC_EXTENSION = True

RUN_NAIVE_BAYES = True

RUN_SEQUENCE_MODEL = False

RUN_WANDB = True

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "ledgar-clause-classification")

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip() or None

WANDB_MODE = os.environ.get("WANDB_MODE", "online")

WANDB_LOG_ARTIFACTS = True

WANDB_LOG_TEXT_TABLES = False

WANDB_LOG_MODEL_FILES = False

TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"

OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

MAX_TRANSFORMER_LENGTH = 256

QWEN_EVAL_SAMPLE_SIZE = 200

QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1


DOWNLOAD_LEDGAR_IF_MISSING = True

DOWNLOAD_CUAD_IF_MISSING = True

USE_HF_CACHE = True

FORCE_REDOWNLOAD = False

paths = build_project_paths(PROJECT_ROOT)
DEVICE = seed_everything(SEED)


In [ ]:
print(f"Project root: {paths.project_root}")
print(f"Colab runtime: {IN_COLAB}")
print(f"Raw LEDGAR directory: {paths.ledgar_raw_dir}")
print(f"Results directory: {paths.results_dir}")
print(f"Device: {DEVICE}")

In [ ]:
# Locate project root.
try:
    PROJECT_ROOT_CHECK = Path(paths.project_root)
except NameError:
    PROJECT_ROOT_CHECK = Path.cwd()

REPORT_TEX = Path(r"C:\Users\ybenj\Downloads\report.tex")

# In Colab, this path only exists if report.tex was uploaded or synced.
# Standard report-facing artifacts are checked either way.
print(f"Project root: {PROJECT_ROOT_CHECK}")
print(f"report.tex found: {REPORT_TEX.exists()} -> {REPORT_TEX}")

# Required report-facing tables and files.
required_files = [
    "outputs/data_summary.csv",
    "outputs/label_distribution.csv",
    "outputs/main_results.csv",
    "outputs/per_class_results.csv",
    "outputs/confusion_pairs.csv",
    "outputs/misclassified_examples.csv",
    "outputs/hyperparameters.csv",
    "outputs/environment.json",
    "outputs/report_artifact_manifest.json",
    "data/processed/dataset_summary.json",
    "data/processed/label_counts.json",
    "data/processed/label_names.txt",
    "data/processed/ledgar_train.jsonl",
    "data/processed/ledgar_validation.jsonl",
    "data/processed/ledgar_test.jsonl",
]

# Conditional model evidence may contain skipped-status rows.
optional_but_expected_files = [
    "outputs/transformer_results.csv",
    "outputs/transformer_predictions.csv",
    "outputs/qwen_results.csv",
    "outputs/qwen_predictions.csv",
    "outputs/qwen_invalid_outputs.csv",
    "outputs/qwen_prompt_examples.txt",
    "results/transformer/runtime.json",
    "results/transformer/training_args.json",
    "results/transformer/training_log_history.json",
    "results/qwen/runtime.json",
    "results/qwen/qwen_run_config.json",
]

# Figures expected by the current report.
standard_report_figures = [
    "figures/label_distribution.png",
    "figures/clause_length_distribution.png",
    "figures/agentic_review_workflow.png",
    "figures/model_comparison_macro_f1.png",
    "figures/confusion_matrix_best_model.png",
    "figures/qwen_invalid_predictions.png",
]

# Add figure refs from report.tex when available.
figure_refs_from_tex = []
if REPORT_TEX.exists():
    tex = REPORT_TEX.read_text(encoding="utf-8", errors="ignore")
    figure_refs_from_tex = re.findall(r"\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}", tex)

figure_refs = sorted(set(standard_report_figures + figure_refs_from_tex))

def file_status(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    exists = path.exists()
    size = path.stat().st_size if exists and path.is_file() else 0
    return {
        "path": relative_path,
        "exists": exists,
        "non_empty": bool(exists and size > 0),
        "size_bytes": size,
    }

def csv_rows(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return len(pd.read_csv(path))
    except Exception:
        return "unreadable"

def jsonl_rows(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    if not path.exists() or path.stat().st_size == 0:
        return None
    return sum(1 for line in path.open("r", encoding="utf-8") if line.strip())

# Check files.
required_status = pd.DataFrame([file_status(p) for p in required_files])
optional_status = pd.DataFrame([file_status(p) for p in optional_but_expected_files])

required_status["rows_if_csv"] = required_status["path"].apply(lambda p: csv_rows(p) if p.endswith(".csv") else None)
optional_status["rows_if_csv"] = optional_status["path"].apply(lambda p: csv_rows(p) if p.endswith(".csv") else None)

print("\nREQUIRED FILES")
display(required_status)

print("\nOPTIONAL / CONDITIONAL MODEL EVIDENCE FILES")
display(optional_status)

# Check figures in report and output locations.
figure_rows = []
for ref in figure_refs:
    ref_path = Path(ref)
    candidates = [
        PROJECT_ROOT_CHECK / ref,
        PROJECT_ROOT_CHECK / "outputs" / ref,
    ]
    # Also check outputs/figures/foo.png for figures/foo.png refs.
    if len(ref_path.parts) >= 2 and ref_path.parts[0] == "figures":
        candidates.append(PROJECT_ROOT_CHECK / "outputs" / "figures" / ref_path.name)

    existing = [p for p in candidates if p.exists() and p.is_file() and p.stat().st_size > 0]
    figure_rows.append({
        "figure_ref": ref,
        "found": bool(existing),
        "found_at": str(existing[0]) if existing else "",
        "size_bytes": existing[0].stat().st_size if existing else 0,
    })

figure_status = pd.DataFrame(figure_rows)
print("\nFIGURES")
display(figure_status)

# Check prediction files against processed test size.
test_rows = jsonl_rows("data/processed/ledgar_test.jsonl")
prediction_dir = PROJECT_ROOT_CHECK / "outputs" / "predictions"
prediction_rows = []

if prediction_dir.exists():
    for pred_path in sorted(prediction_dir.glob("*_test_predictions.jsonl")):
        count = sum(1 for line in pred_path.open("r", encoding="utf-8") if line.strip())
        prediction_rows.append({
            "prediction_file": str(pred_path.relative_to(PROJECT_ROOT_CHECK)),
            "rows": count,
            "matches_test_rows": count == test_rows,
            "expected_test_rows": test_rows,
        })

prediction_status = pd.DataFrame(prediction_rows)
print("\nPREDICTION ROW COUNTS")
display(prediction_status)

# Final pass/fail summary.
missing_required = required_status[~required_status["non_empty"]]
missing_figures = figure_status[~figure_status["found"]]
bad_predictions = prediction_status[prediction_status["matches_test_rows"] == False] if not prediction_status.empty else pd.DataFrame()

print("\nSUMMARY")
print(f"Required files OK: {missing_required.empty}")
print(f"Figures OK: {missing_figures.empty}")
print(f"Prediction row counts OK: {bad_predictions.empty}")
print(f"Processed test rows: {test_rows}")

if not missing_required.empty:
    print("\nMissing/empty required files:")
    display(missing_required)

if not missing_figures.empty:
    print("\nMissing figures:")
    display(missing_figures)

if not bad_predictions.empty:
    print("\nPrediction files with wrong row counts:")
    display(bad_predictions)


In [ ]:

# Final report evidence audit cell
# Run before using the notebook outputs to fill report.tex.


# In Colab, set this manually if report.tex is uploaded or synced.
# REPORT_TEX_PATH = "/content/drive/MyDrive/.../report.tex"
REPORT_TEX_PATH = globals().get("REPORT_TEX_PATH", r"C:\Users\ybenj\Downloads\report.tex")

try:
    PROJECT_ROOT_CHECK = Path(paths.project_root)
except NameError:
    PROJECT_ROOT_CHECK = Path.cwd()

REPORT_TEX = Path(REPORT_TEX_PATH)

print(f"Project root: {PROJECT_ROOT_CHECK}")
print(f"report.tex found: {REPORT_TEX.exists()} -> {REPORT_TEX}")


def safe_display(title, df):
    print(f"\n{title}")
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def file_mtime(path):
    if not path.exists():
        return None
    return datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc)


def count_jsonl(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    return sum(1 for line in path.open("r", encoding="utf-8") if line.strip())


def read_csv_safe(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Could not read CSV {path}: {type(exc).__name__}: {exc}")
        return None


def read_json_safe(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Could not read JSON {path}: {type(exc).__name__}: {exc}")
        return None


def resolve_report_path(value):
    if pd.isna(value) or not str(value).strip():
        return None

    raw = str(value)
    direct = Path(raw)
    if direct.exists():
        return direct

    # Handle paths saved in Colab like /content/drive/.../results/...
    for marker in ["outputs/", "results/", "figures/", "data/processed/"]:
        if marker in raw.replace("\\", "/"):
            rel = raw.replace("\\", "/").split(marker, 1)[1]
            candidate = PROJECT_ROOT_CHECK / marker.rstrip("/") / rel
            if candidate.exists():
                return candidate

    candidate = PROJECT_ROOT_CHECK / raw
    if candidate.exists():
        return candidate

    return None


def safe_name(value):
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(value).lower()).strip("_") or "item"


# 1. Required artifact checks

required_specs = {
    "outputs/data_summary.csv": ["metric", "value", "notes"],
    "outputs/label_distribution.csv": ["label", "label_id", "train_count", "validation_count", "test_count", "total_count"],
    "outputs/main_results.csv": ["model_family", "model_name", "sample_size", "accuracy", "macro_f1", "weighted_f1"],
    "outputs/per_class_results.csv": ["model_family", "model_name", "label", "precision", "recall", "f1_score", "support"],
    "outputs/confusion_pairs.csv": ["rank", "true_label", "predicted_label", "count"],
    "outputs/misclassified_examples.csv": ["text", "label", "predicted_label"],
    "outputs/hyperparameters.csv": ["component", "parameter", "value"],
    "outputs/environment.json": None,
    "outputs/leakage_audit.json": None,
    "outputs/report_artifact_manifest.json": None,
    "data/processed/dataset_summary.json": None,
    "data/processed/label_counts.json": None,
    "data/processed/label_names.txt": None,
    "data/processed/ledgar_train.jsonl": None,
    "data/processed/ledgar_validation.jsonl": None,
    "data/processed/ledgar_test.jsonl": None,
    "results/final_model_comparison.csv": ["model_family", "model_name", "sample_size", "accuracy", "macro_f1", "weighted_f1"],
}

artifact_rows = []
for rel_path, expected_cols in required_specs.items():
    path = PROJECT_ROOT_CHECK / rel_path
    exists = path.exists()
    non_empty = exists and path.is_file() and path.stat().st_size > 0
    columns_ok = True
    rows = None
    missing_cols = []

    if non_empty and rel_path.endswith(".csv"):
        df = read_csv_safe(path)
        if df is not None:
            rows = len(df)
            missing_cols = [c for c in expected_cols if c not in df.columns]
            columns_ok = not missing_cols
        else:
            columns_ok = False

    if non_empty and rel_path.endswith(".jsonl"):
        rows = count_jsonl(path)

    artifact_rows.append({
        "path": rel_path,
        "exists": exists,
        "non_empty": non_empty,
        "rows": rows,
        "columns_ok": columns_ok,
        "missing_columns": ", ".join(missing_cols),
        "modified_utc": file_mtime(path),
    })

artifact_status = pd.DataFrame(artifact_rows)
safe_display("1. REQUIRED ARTIFACTS", artifact_status)


# 2. Split consistency checks

split_paths = {
    "train": PROJECT_ROOT_CHECK / "data/processed/ledgar_train.jsonl",
    "validation": PROJECT_ROOT_CHECK / "data/processed/ledgar_validation.jsonl",
    "test": PROJECT_ROOT_CHECK / "data/processed/ledgar_test.jsonl",
}
split_counts = {split: count_jsonl(path) for split, path in split_paths.items()}

data_summary = read_csv_safe(PROJECT_ROOT_CHECK / "outputs/data_summary.csv")
summary_counts = {}

if data_summary is not None:
    summary_map = dict(zip(data_summary["metric"], data_summary["value"]))
    for split in ["train", "validation", "test"]:
        key = f"filtered_{split}_examples"
        try:
            summary_counts[split] = int(float(summary_map.get(key)))
        except Exception:
            summary_counts[split] = None

split_status = pd.DataFrame([
    {
        "split": split,
        "processed_jsonl_rows": split_counts.get(split),
        "data_summary_rows": summary_counts.get(split),
        "matches_data_summary": split_counts.get(split) == summary_counts.get(split),
    }
    for split in ["train", "validation", "test"]
])
safe_display("2. SPLIT CONSISTENCY", split_status)


# ----------------------------
# 3. Leakage audit check
# ----------------------------

leakage = read_json_safe(PROJECT_ROOT_CHECK / "outputs/leakage_audit.json")
leakage_rows = []

if leakage:
    after = leakage.get("cross_split_overlaps_after_deduplication", {})
    for pair, values in after.items():
        leakage_rows.append({
            "pair": pair,
            "text_overlap": values.get("text_overlap"),
            "text_label_overlap": values.get("text_label_overlap"),
            "ok_zero_overlap": values.get("text_overlap") == 0 and values.get("text_label_overlap") == 0,
        })

leakage_status = pd.DataFrame(leakage_rows)
safe_display("3. LEAKAGE AUDIT", leakage_status)


# ----------------------------
# 4. Metrics consistency checks
# ----------------------------

main_results = read_csv_safe(PROJECT_ROOT_CHECK / "outputs/main_results.csv")
final_comparison = read_csv_safe(PROJECT_ROOT_CHECK / "results/final_model_comparison.csv")

metric_consistency_rows = []

if main_results is not None and final_comparison is not None:
    keys = ["model_family", "model_name"]
    metric_cols = ["sample_size", "accuracy", "macro_f1", "weighted_f1"]
    merged = main_results[keys + metric_cols].merge(
        final_comparison[keys + metric_cols],
        on=keys,
        suffixes=("_outputs", "_results"),
        how="outer",
        indicator=True,
    )

    for _, row in merged.iterrows():
        ok = row["_merge"] == "both"
        diffs = []
        if ok:
            for col in metric_cols:
                left = row[f"{col}_outputs"]
                right = row[f"{col}_results"]
                if pd.isna(left) and pd.isna(right):
                    continue
                if col == "sample_size":
                    same = int(left) == int(right)
                else:
                    same = abs(float(left) - float(right)) < 1e-9
                if not same:
                    ok = False
                    diffs.append(col)

        metric_consistency_rows.append({
            "model_family": row.get("model_family"),
            "model_name": row.get("model_name"),
            "present_in_both": row["_merge"] == "both",
            "metrics_match": ok,
            "different_columns": ", ".join(diffs),
        })

metric_consistency = pd.DataFrame(metric_consistency_rows)
safe_display("4. METRIC CONSISTENCY: outputs/main_results.csv vs results/final_model_comparison.csv", metric_consistency)


# ----------------------------
# 5. Classification report/confusion matrix existence
# ----------------------------

report_rows = []

if main_results is not None:
    for _, row in main_results.iterrows():
        sample_size = row.get("sample_size")
        macro_f1 = row.get("macro_f1")
        is_completed = pd.notna(macro_f1) and pd.notna(sample_size) and int(sample_size) > 0

        report_path = resolve_report_path(row.get("classification_report_path"))
        cm_path = resolve_report_path(row.get("confusion_matrix_path"))

        report_rows.append({
            "model_name": row.get("model_name"),
            "completed_result": is_completed,
            "classification_report_found": bool(report_path) if is_completed else "not_required_if_skipped",
            "confusion_matrix_found": bool(cm_path) if is_completed else "not_required_if_skipped",
            "classification_report_path": str(report_path) if report_path else "",
            "confusion_matrix_path": str(cm_path) if cm_path else "",
        })

report_status = pd.DataFrame(report_rows)
safe_display("5. REPORT + CONFUSION MATRIX EVIDENCE", report_status)


# ----------------------------
# 6. Prediction file row counts
# ----------------------------

test_rows = split_counts.get("test")
prediction_rows = []
prediction_dir = PROJECT_ROOT_CHECK / "outputs/predictions"

if prediction_dir.exists():
    for pred_path in sorted(prediction_dir.glob("*_test_predictions.jsonl")):
        rows = count_jsonl(pred_path)
        prediction_rows.append({
            "prediction_file": str(pred_path.relative_to(PROJECT_ROOT_CHECK)),
            "rows": rows,
            "expected_test_rows": test_rows,
            "matches_test_rows": rows == test_rows,
            "modified_utc": file_mtime(pred_path),
        })

prediction_status = pd.DataFrame(prediction_rows)
safe_display("6. PREDICTION ROW COUNTS", prediction_status)


# ----------------------------
# 7. Skipped transformer/Qwen guard
# ----------------------------

guard_rows = []

if main_results is not None:
    for model_group, mask in {
        "transformer": main_results["model_family"].astype(str).str.contains("transformer", case=False, na=False)
                       | main_results["model_name"].astype(str).str.contains("bert|distilbert|legal", case=False, na=False),
        "qwen": main_results["model_name"].astype(str).str.contains("qwen", case=False, na=False)
                | main_results["model_family"].astype(str).str.contains("qwen|llm", case=False, na=False),
    }.items():
        subset = main_results[mask]
        if subset.empty:
            guard_rows.append({
                "model_group": model_group,
                "present": False,
                "real_result_rows": 0,
                "skipped_or_empty_rows": 0,
                "safe_to_claim_results": False,
            })
        else:
            real = subset[pd.to_numeric(subset["sample_size"], errors="coerce").fillna(0).gt(0) & subset["macro_f1"].notna()]
            guard_rows.append({
                "model_group": model_group,
                "present": True,
                "real_result_rows": len(real),
                "skipped_or_empty_rows": len(subset) - len(real),
                "safe_to_claim_results": len(real) > 0,
            })

skipped_guard = pd.DataFrame(guard_rows)
safe_display("7. SKIPPED MODEL GUARD", skipped_guard)


# ----------------------------
# 8. Figure reference checks
# ----------------------------

standard_figures = [
    "figures/label_distribution.png",
    "figures/clause_length_distribution.png",
    "figures/agentic_review_workflow.png",
    "figures/model_comparison_macro_f1.png",
    "figures/confusion_matrix_best_model.png",
    "figures/qwen_invalid_predictions.png",
]

figure_refs_from_tex = []
todo_rows = []

if REPORT_TEX.exists():
    tex = REPORT_TEX.read_text(encoding="utf-8", errors="ignore")
    figure_refs_from_tex = re.findall(r"\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}", tex)

    for line_no, line in enumerate(tex.splitlines(), 1):
        for todo in re.findall(r"\\todo\{([^}]*)\}", line):
            todo_rows.append({
                "line": line_no,
                "todo": todo,
                "likely_evidence": (
                    "outputs/main_results.csv" if any(x in todo.lower() for x in ["result", "score", "model", "f1"]) else
                    "outputs/data_summary.csv" if any(x in todo.lower() for x in ["example", "length", "train", "validation", "test"]) else
                    "outputs/per_class_results.csv" if "class" in todo.lower() or "categor" in todo.lower() else
                    "outputs/confusion_pairs.csv" if "label" in todo.lower() else
                    "manual review needed"
                ),
            })

figure_refs = sorted(set(standard_figures + figure_refs_from_tex))

qwen_real = False
if main_results is not None:
    qwen_mask = main_results["model_name"].astype(str).str.contains("qwen", case=False, na=False)
    qwen_subset = main_results[qwen_mask]
    if not qwen_subset.empty:
        qwen_real = bool(
            pd.to_numeric(qwen_subset["sample_size"], errors="coerce").fillna(0).gt(0).any()
            and qwen_subset["macro_f1"].notna().any()
        )

figure_rows = []
for ref in figure_refs:
    ref_path = Path(ref)
    candidates = [
        PROJECT_ROOT_CHECK / ref,
        PROJECT_ROOT_CHECK / "outputs" / ref,
    ]
    if len(ref_path.parts) >= 2 and ref_path.parts[0] == "figures":
        candidates.append(PROJECT_ROOT_CHECK / "outputs" / "figures" / ref_path.name)

    existing = [p for p in candidates if p.exists() and p.is_file() and p.stat().st_size > 0]
    is_qwen_invalid = ref_path.name == "qwen_invalid_predictions.png"

    figure_rows.append({
        "figure_ref": ref,
        "found": bool(existing),
        "conditional_qwen_figure": is_qwen_invalid,
        "qwen_real_result_available": qwen_real,
        "ok_for_pipeline": bool(existing) or (is_qwen_invalid and not qwen_real),
        "report_edit_warning": "remove/keep TODO if Qwen skipped" if is_qwen_invalid and not qwen_real else "",
        "found_at": str(existing[0]) if existing else "",
        "size_bytes": existing[0].stat().st_size if existing else 0,
    })

figure_status = pd.DataFrame(figure_rows)
safe_display("8. FIGURE REFERENCES", figure_status)


# ----------------------------
# 9. TODO evidence map
# ----------------------------

todo_status = pd.DataFrame(todo_rows)
safe_display("9. REPORT TODO EVIDENCE MAP", todo_status)


# ----------------------------
# 10. Final strict summary
# ----------------------------

critical_failures = []

if not artifact_status["non_empty"].all():
    critical_failures.append("Missing or empty required artifacts.")

if not artifact_status["columns_ok"].all():
    critical_failures.append("Some required CSVs are missing expected columns.")

if not split_status["matches_data_summary"].all():
    critical_failures.append("Processed split counts do not match outputs/data_summary.csv.")

if not leakage_status.empty and not leakage_status["ok_zero_overlap"].all():
    critical_failures.append("Leakage audit still shows cross-split overlaps.")

if not metric_consistency.empty and not metric_consistency["metrics_match"].all():
    critical_failures.append("outputs/main_results.csv and results/final_model_comparison.csv disagree.")

if not prediction_status.empty and not prediction_status["matches_test_rows"].all():
    critical_failures.append("One or more test prediction files do not match processed test row count.")

completed_missing_reports = report_status[
    (report_status["completed_result"] == True)
    & (
        (report_status["classification_report_found"] != True)
        | (report_status["confusion_matrix_found"] != True)
    )
]
if not completed_missing_reports.empty:
    critical_failures.append("A completed model is missing a classification report or confusion matrix.")

blocking_figures = figure_status[figure_status["ok_for_pipeline"] != True]
if not blocking_figures.empty:
    critical_failures.append("One or more required non-conditional figures are missing.")

print("\nFINAL SUMMARY")
print(f"Critical failures: {len(critical_failures)}")
for failure in critical_failures:
    print(f"- {failure}")

if not critical_failures:
    print("PASS: report-facing evidence looks internally consistent.")
else:
    print("FAIL: fix the issues above before asking Codex to fill report.tex.")
